In [1]:
!pip install langchain langchain-community langchain-text-splitters \
             langchain-core sentence-transformers faiss-cpu pypdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

# Replace 'BFL' with your actual folder name exactly as it appears in Drive
folder_path = '/content/drive/MyDrive/BFL'

# Confirm it exists and see what's inside
os.listdir(folder_path)

['AI & IP in Education_October 2025.pdf',
 'Alberta Colleges - 2025 Symposium - FM Presentation (final).pdf',
 'ACUTI Claims 101 Oct 2025.pdf',
 'AB Colleges 2025 Symposium - Edmonton October 2025 - Presentation.pdf']

In [4]:
# LOADER
from langchain_community.document_loaders import PyPDFDirectoryLoader

# CHUNKER — moved to langchain_text_splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

# EMBEDDER + VECTOR STORE
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# GENERATOR
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

/tmp/ipykernel_949/2897637619.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


In [5]:
# Load all PDFs from your Drive folder
loader = PyPDFDirectoryLoader(folder_path)
docs = loader.load()
print(f"Loaded {len(docs)} pages")

# Chunk them
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(f"Created {len(chunks)} chunks")

Loaded 146 pages
Created 237 chunks


In [19]:
# Embed and store in FAISS
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 12})

print("Vector store ready")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store ready


In [20]:
# Prompt template
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}

Question: {question}
""")

# Format retrieved docs into a single string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [11]:
!pip install langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.1 MB/s eta 0:00:00


In [18]:
from google.colab import userdata
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=userdata.get('grokapi')
)

print("LLM ready")

LLM ready


In [21]:
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}

Question: {question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer = chain.invoke("what is covered for water damage?with the source ")
print(answer)

According to the provided context, water damage is covered with a deductible of 5% minimum $50,000, subject to a maximum of $150,000. This is mentioned on page 13 of the BFL CANADA Risk and Insurance Services Inc. document:

"- 5% minimum $50,000 subject to maximum $150,000 Water Damage"
